In [1]:
# Agents in LlamaIndex
#
# Initialising agents
#
# Let's start by initialising an agent. We will use the basic `AgentWorkflow` class to create an agent.
#

from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult, AgentStream
#from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI


def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a / b

#
# using Ollama LLM
#

from llama_index.llms.ollama import Ollama

QWEN25_CODER_14B_Q4_K_M = "qwen2.5-coder:14b"

MODEL = QWEN25_CODER_14B_Q4_K_M

llm = llm = Ollama(
    base_url="http://localhost:11434",
    model=MODEL,
    timeout=0
)
# llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[subtract, multiply, divide, add],
    llm=llm,
    system_prompt="You are a math agent that can add, subtract, multiply, and divide numbers using provided tools.",
)

In [2]:
# Then, we can run the agent and get the response and reasoning behind the tool calls.

if not agent:
    raise ValueError('agent object not set')

handler = agent.run("What is (2 + 2) * 2?")

import nest_asyncio
nest_asyncio.apply()  # This is needed to run the query engine

async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp


Called tool:  add {'a': 2, 'b': 2} => 4

Called tool:  multiply {'a': 4, 'b': 2} => 8

Called tool:  add {'a': 2, 'b': 2} => 4

Called tool:  multiply {'a': 4, 'b': 2} => 8
The result of (2 + 2) * 2 is 8.

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': [], 'thinking': ''}, blocks=[TextBlock(block_type='text', text='The result of (2 + 2) * 2 is 8.')]), tool_calls=[ToolCallResult(tool_name='add', tool_kwargs={'a': 2, 'b': 2}, tool_id='add', tool_output=ToolOutput(content='4', tool_name='add', raw_input={'args': (), 'kwargs': {'a': 2, 'b': 2}}, raw_output=4, is_error=False), return_direct=False), ToolCallResult(tool_name='multiply', tool_kwargs={'a': 4, 'b': 2}, tool_id='multiply', tool_output=ToolOutput(content='8', tool_name='multiply', raw_input={'args': (), 'kwargs': {'a': 4, 'b': 2}}, raw_output=8, is_error=False), return_direct=False), ToolCallResult(tool_name='add', tool_kwargs={'a': 2, 'b': 2}, tool_id='add', tool_output=ToolOutput(content='4', tool_name='add', raw_input={'args': (), 'kwargs': {'a': 2, 'b': 2}}, raw_output=4, is_error=False), return_direct=False), ToolCallResult(tool_name='multiply', tool_kwargs={'a': 4, 

In [3]:
# In a similar fashion, we can pass state and context to the agent.

if not agent:
    raise ValueError('agent object not set')

from llama_index.core.workflow import Context

ctx = Context(agent)

import nest_asyncio
nest_asyncio.apply()  # This is needed to run the query engine

response = await agent.run("My name is Bob.", ctx=ctx)
response = await agent.run("What was my name again?", ctx=ctx)
response

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': [], 'thinking': ''}, blocks=[TextBlock(block_type='text', text='Your name is Bob.')]), tool_calls=[], raw={'model': 'qwen2.5-coder:14b', 'created_at': '2025-06-22T19:43:25.222833999Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4819531760, 'load_duration': 22476630, 'prompt_eval_count': 539, 'prompt_eval_duration': 2796106123, 'eval_count': 6, 'eval_duration': 1946910142, 'message': Message(role='assistant', content='', thinking=None, images=None, tool_calls=None), 'usage': {'prompt_tokens': 539, 'completion_tokens': 6, 'total_tokens': 545}}, current_agent_name='Agent')

In [30]:
# Creating RAG Agents with QueryEngineTools
#
# Let's now re-use the `QueryEngine` we defined in the [previous unit on tools](/tools.ipynb) and convert it into a `QueryEngineTool`. 
# We will pass it to the `AgentWorkflow` class to create a RAG agent.
#

import chromadb

from llama_index.core import VectorStoreIndex
#from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
#from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore

# Create a vector store
db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

#
# using Ollama embedding 
#

from llama_index.embeddings.ollama import OllamaEmbedding

BGE_SMALL_EN_V15_Q4_K_M = "qllama/bge-small-en-v1.5:q4_k_m"

MODEL = BGE_SMALL_EN_V15_Q4_K_M

embed_model = OllamaEmbedding(
    model_name=MODEL,
    base_url="http://localhost:11434",
    ollama_additional_kwargs={"mirostat": 0},
)
#embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")


#
# using Ollama LLM
#

from llama_index.llms.ollama import Ollama

QWEN25_CODER_14B_Q4_K_M = "qwen2.5-coder:14b"

MODEL = QWEN25_CODER_14B_Q4_K_M

llm = llm = Ollama(
    base_url="http://localhost:11434",
    model=MODEL,
)
#llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")


# Create a query engine
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model, timeout=int(sys.maxsize)
)


import sys

import nest_asyncio
nest_asyncio.apply()  # This is needed to run the query engine
query_engine = index.as_query_engine(
    llm=llm,
    timeout=int(sys.maxsize),
    retry_policy=5,
    delay=30,

)

query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="personas",
    description="descriptions for various types of personas",
    return_direct=False,
)

# Create a RAG agent
query_engine_agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[query_engine_tool],
    llm=llm,
    system_prompt="You are a helpful assistant that has access to a database containing persona descriptions. ",
)

In [31]:
# And, we can once more get the response and reasoning behind the tool calls.

import sys

if not query_engine_agent:
    raise ValueError('query_engine_agent object not set')

import nest_asyncio
nest_asyncio.apply()  # This is needed to run the query engine
handler = query_engine_agent.run(
    "Search the database for 'science fiction' and return some persona descriptions.",
    timeout=int(sys.maxsize),
    retry_policy=5,
    delay=30,
)

nest_asyncio.apply()  # This is needed to run the query engine
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

nest_asyncio.apply()  # This is needed to run the query engine
resp = await handler
resp


Called tool:  personas {'input': 'science fiction'} => 


WorkflowHandler exception was never retrieved
future: <WorkflowHandler finished exception=WorkflowRuntimeError("Error in step 'run_agent_step': ")>
Traceback (most recent call last):
  File "/home/mikejay/pythonVenvDirs/hfenv/lib/python3.12/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/home/mikejay/pythonVenvDirs/hfenv/lib/python3.12/site-packages/httpx/_transports/default.py", line 394, in handle_async_request
    resp = await self._pool.handle_async_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mikejay/pythonVenvDirs/hfenv/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 256, in handle_async_request
    raise exc from None
  File "/home/mikejay/pythonVenvDirs/hfenv/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 236, in handle_async_request
    response = await connection.handle_async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fi

WorkflowRuntimeError: Error in step 'run_agent_step': 

In [32]:
# Creating multi-agent systems
#
# We can also create multi-agent systems by passing multiple agents to the `AgentWorkflow` class.
#

if not llm:
    raise ValueError('llm object not set')


from llama_index.core.agent.workflow import (
    AgentWorkflow,
    ReActAgent,
)


# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b


# Create agent configs
# NOTE: we can use FunctionAgent or ReActAgent here.
# FunctionAgent works for LLMs with a function calling API.
# ReActAgent works for any LLM.
calculator_agent = ReActAgent(
    name="calculator",
    description="Performs basic arithmetic operations",
    system_prompt="You are a calculator assistant. Use your tools for any math operation.",
    tools=[add, subtract],
    llm=llm,
)

query_agent = ReActAgent(
    name="info_lookup",
    description="Looks up information about XYZ",
    system_prompt="Use your tool to query a RAG system to answer information about XYZ",
    tools=[query_engine_tool],
    llm=llm,
)

# Create and run the workflow
agent = AgentWorkflow(agents=[calculator_agent, query_agent], root_agent="calculator")

# Run the system
handler = agent.run(user_msg="Can you add 5 and 3?")

#
# combined cells
#

async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

WorkflowRuntimeError: Error in step 'run_agent_step': 